# Create Agent

In [ ]:
from typing import Annotated, Literal

from IPython.display import Image
from langchain.agents import AgentState, create_agent
from langchain.messages import ToolMessage
from langchain.tools import InjectedToolCallId, tool
from langgraph.types import Command
from loguru import logger

from chain_reaction.config import get_chat_model
from chain_reaction.reducers import reduce_list
from chain_reaction.utils import format_messages

## Tools

In [ ]:
type Numeric = int | float
type Operation = Literal["add", "subtract", "multiply", "divide"]


@tool
def calculator(  # noqa: C901
    operation: Operation,
    a: Numeric,
    b: Numeric,
) -> Numeric | str:
    """Apply an arithmetic operation to two numbers and return result.

    Args:
        operation (Operation): Operation to apply.
        a (Numeric): First number
        b (Numeric): Second number

    Returns:
        Numeric | str: Result of arithmetic operation applied to the two input numbers, or error message.
    """
    logger.info("calculator: {operation}({a}, {b})", operation=operation, a=a, b=b)
    match operation:
        case "add":
            return a + b
        case "subtract":
            return a - b
        case "multiply":
            return a * b
        case "divide":
            if b == 0:
                return "error: Cannot divide by 0."
            return a / b
        case _:
            return f"error: {operation} not supported."

In [ ]:
calculator.invoke({"operation": "add", "a": 1, "b": 2})
calculator.invoke({"operation": "divide", "a": 1, "b": 0})

## Define ReAct agent

In [ ]:
SYSTEM_PROMPT = """
You are a calculator agent who uses your tools to help users with arithmetic calculations.
Always use your tools to perform calculations, never make up the results.
Return all text as plain text WITHOUT Markdown math delimiters.
"""

llm_model = get_chat_model()

agent = create_agent(
    model=llm_model,
    tools=[calculator],
    system_prompt=SYSTEM_PROMPT,
).with_config({"recursion_limit": 20})

# Display agent graph
Image(agent.get_graph().draw_mermaid_png())

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "what is 3.1 * 4.2?"}]})
format_messages(result.get("messages", []))

## Agent w/ Custom State
- The (agent) graph has a typed data structure that is available to each node for the duration of the graph and can be persisted in long-term storage. 
- You can use this to store information to share between nodes
- When you define state for a graph, you define the data types and a 'reducer' function. The reducer describes how information is added to that element.
- The LLM model does NOT have access to the state
- To give a tool access to the state instance, we inject the state into the tool (after the LLM has made the tool call)

Imagine we want to keep track of tool class made as the agent loops (maybe to measure progress against a todo list). We can achieve that by adding a `calc_ops` list to the `AgentState`.

In [ ]:
class CalculationOpsState(AgentState):
    """Custom agent state that tracks history of calculation of operations made by agent."""

    calc_ops: Annotated[list[str], reduce_list]


@tool
def calculator_with_state(  # noqa: C901
    operation: Operation,
    a: Numeric,
    b: Numeric,
    tool_call_id: Annotated[str, InjectedToolCallId],  # not sent to LLM
) -> Command:
    """Apply an arithmetic operation to two numbers and return result.

    Args:
        operation (Operation): Operation to apply.
        a (Numeric): First number
        b (Numeric): Second number
        tool_call_id (Annotated[str, InjectedToolCallId]): Tool call id.

    Returns:
       Command: State update command.
    """
    logger.info("calculator: {operation}({a}, {b})", operation=operation, a=a, b=b)

    result: int | float | str
    match operation:
        case "add":
            result = a + b
        case "subtract":
            result = a - b
        case "multiply":
            result = a * b
        case "divide":
            if b == 0:
                result = "error: Cannot divide by 0."
            result = a / b
        case _:
            result = f"error: {operation} not supported."

    return Command(
        update={
            "calc_ops": [f"{operation}({a}, {b})"],  # Append calculation operation to list
            "messages": [
                ToolMessage(str(result), tool_call_id=tool_call_id)  # Result of tool call
            ],
        }
    )


agent_with_state = create_agent(
    model=llm_model,
    tools=[calculator_with_state],
    system_prompt=SYSTEM_PROMPT,
    state_schema=CalculationOpsState,  # Supply customized state schema here
).with_config({"recursion_limit": 20})

In [ ]:
result = agent_with_state.invoke({
    "messages": [{"role": "user", "content": "what is 3.1 * 4.2 + 5.9 / 6.7 - 13.2 * 0.652?"}]
})
calc_ops = result.get("calc_ops", [])
format_messages(result.get("messages", []))
print(f"Calculation operation sequence: {calc_ops}")